# MT-SEC Filtering and Selection

This notebook filters the MT-SEC dataset to select 20 samples under constraints:
- 10 PASS: capability=1, safety=1
- 10 FAIL: capability=1, safety=0
- Singleturn and one multiturn interaction (expansion/editing/refactor) share the same ID and labels
- Both selected singleturn and selected multiturn token counts are < 8000
- Random, reproducible pick of the multiturn type per ID (seeded)

Outputs:
- Printed lists in the requested format
- Saved JSON at `/space3/yangfc/benchmarks/MT-SEC/filtered_selection.json`


In [1]:
import json
from pathlib import Path
import random

TOKEN_LIMIT = 8000
SEED = 42
BASE_DIR = Path('/space3/yangfc/benchmarks/MT-SEC')
DATA_PATH = BASE_DIR / 'transformed-secode-allsamples.json'
OUTPUT_PATH = BASE_DIR / 'filtered_selection.json'
print('Using data from:', DATA_PATH)
assert DATA_PATH.exists(), 'Missing transformed-secode-allsamples.json'


Using data from: /space3/yangfc/benchmarks/MT-SEC/transformed-secode-allsamples.json


In [2]:
def label_tuple(result_dict):
    if not isinstance(result_dict, dict):
        return None
    cap = result_dict.get('capability')
    saf = result_dict.get('safety')
    if cap is None or saf is None or cap == '' or saf == '':
        return None
    return (str(cap), str(saf))

def pick_ids(data, token_limit=TOKEN_LIMIT, seed=SEED):
    random.seed(seed)
    candidates_pass = []  # (id, multiturn_type)
    candidates_fail = []

    for sid, sample in data.items():
        results = sample.get('results', {})
        tokens = sample.get('tokens', {})
        st_labels = label_tuple(results.get('singleturn'))
        if not st_labels:
            continue
        st_cap, st_saf = st_labels
        if st_cap != '1':
            continue
        st_tokens = tokens.get('singleturn')
        if st_tokens is None or st_tokens >= token_limit:
            continue
        eligible_types = []
        for mt_type in ('expansion', 'editing', 'refactor'):
            mt_res = results.get(mt_type)
            mt_labels = label_tuple(mt_res)
            if not mt_labels or mt_labels != st_labels:
                continue
            mt_tokens = tokens.get(mt_type)
            if mt_tokens is None or mt_tokens >= token_limit:
                continue
            eligible_types.append(mt_type)
        if not eligible_types:
            continue
        mt_choice = random.choice(eligible_types)
        if st_saf == '1':
            candidates_pass.append((sid, mt_choice))
        elif st_saf == '0':
            candidates_fail.append((sid, mt_choice))

    def sample_10(items):
        if len(items) <= 10:
            return items
        return random.sample(items, 10)

    sel_pass = sample_10(candidates_pass)
    sel_fail = sample_10(candidates_fail)

    def build_entries(pairs):
        out = []
        for sid, mt_type in pairs:
            s = data[sid]
            out.append({
                'id': sid,
                'singleturn': 'singleturn',
                'multiturn': mt_type,
                'labels': {
                    'singleturn': label_tuple(s['results']['singleturn']),
                    'multiturn': label_tuple(s['results'][mt_type]),
                },
                'tokens': {
                    'singleturn': s['tokens'].get('singleturn'),
                    mt_type: s['tokens'].get(mt_type),
                },
            })
        return out

    return {
        'pass': build_entries(sel_pass),
        'fail': build_entries(sel_fail),
    }


In [3]:
# Load, run selection, save, and print
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)
selection = pick_ids(data)
with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    json.dump(selection, f, indent=2)

pass_list = [f"{e['id']}: [singleturn, \"{e['multiturn']}\"]" for e in selection['pass']]
fail_list = [f"{e['id']}: [singleturn, \"{e['multiturn']}\"]" for e in selection['fail']]
print('pass: 10 id :', pass_list)
print('fail: 10 id: ', fail_list)
print('Saved to:', OUTPUT_PATH)


pass: 10 id : ['732f2258: [singleturn, "expansion"]', '5b8fb3fc: [singleturn, "refactor"]', '992e49f1: [singleturn, "editing"]', 'd587e3fa: [singleturn, "editing"]', '43b45698: [singleturn, "editing"]', '2296cdab: [singleturn, "editing"]', '201056af: [singleturn, "refactor"]', '440543a2: [singleturn, "refactor"]', '19daaf2f: [singleturn, "expansion"]', 'e13d3c89: [singleturn, "refactor"]']
fail: 10 id:  ['991f7747: [singleturn, "editing"]', '0c191a3a: [singleturn, "expansion"]', '8a9bbae0: [singleturn, "editing"]', '441da129: [singleturn, "editing"]', '4bb4f31d: [singleturn, "editing"]', 'f9074187: [singleturn, "editing"]', 'df1af3d7: [singleturn, "editing"]', 'f8056028: [singleturn, "expansion"]', '10b87c4c: [singleturn, "refactor"]', '4bc6fe9c: [singleturn, "refactor"]']
Saved to: /space3/yangfc/benchmarks/MT-SEC/filtered_selection.json


In [ ]:
# Quick validation of constraints
ok = True
for group in ('pass','fail'):
    for e in selection[group]:
        sid = e['id']
        mt = e['multiturn']
        s = data[sid]
        st = s['results']['singleturn']
        mtres = s['results'][mt]
        stt = s['tokens']['singleturn']
        mtt = s['tokens'][mt]
        if st['capability'] != '1':
            print('cap mismatch', group, sid); ok=False
        if group == 'pass' and st['safety'] != '1':
            print('safety mismatch pass', sid); ok=False
        if group == 'fail' and st['safety'] != '0':
            print('safety mismatch fail', sid); ok=False
        if (st['capability'], st['safety']) != (mtres['capability'], mtres['safety']):
            print('label mismatch', sid, mt); ok=False
        if stt >= TOKEN_LIMIT or mtt >= TOKEN_LIMIT:
            print('token too high', sid, mt, stt, mtt); ok=False
print('OK?', ok)
